# Final Project: Topic Modeling
**Problem Statement:**

You are to develop your own use case for topic modeling. The use case should involve text data that interests you and should be realistic for a data scientist or machine learning developer. 

In [3]:
import pandas as pd
import langdetect  # language detection
import matplotlib.pyplot  # plotting
import nltk  # natural language processing
import numpy  # arrays and matrices
import pandas  # dataframes
import pyLDAvis  # plotting
import pyLDAvis.lda_model  # plotting
import regex  # regular expressions
import sklearn  # machine learning
import unicodedata  # unicode data manipulation

# Text preprocessing and feature extraction
from sklearn.feature_extraction.text import TfidfVectorizer # TF-IDF vectorizer
from nltk.stem import WordNetLemmatizer  # lemmatizer
from sklearn.feature_extraction.text import CountVectorizer  # Count vectorizer

# Download necessary NLTK resources
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\gly3\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\gly3\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\gly3\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


True

In [5]:
url = "https://raw.githubusercontent.com/Hunteracademic/Unsupervised_assignment_1/master/tripadvisor_hotel_reviews.csv"
df = pd.read_csv(url)
df.head()

,Review,Rating
0,nice hotel expensive parking got good deal sta...,4
1,ok nothing special charge diamond member hilto...,2
2,nice rooms not 4* experience hotel monaco seat...,3
3,"unique, great stay, wonderful time hotel monac...",5
4,"great stay great stay, went seahawk game aweso...",5


Executive Summary: In 180 to 200 words, provide an overview of the notebooks you developed. Describe the use case, data, preprocessing steps, model development, and main points of the analysis. State which model works best or that none of the models were satisfactory and provide reasons. Describe the topics and explain how the model will address the use case, or if none of the models worked well, state what the next steps should be.

Preprocessing: Clean and prepare text for LDA and NMF topic modeling. Include steps such as case normalization, lemmatization, stop word removal, and tokenization. 

### Language Filter

In [6]:
def do_language_identifying(txt):
    try: the_language = langdetect.detect(txt)
    except: the_language = 'none'
    return the_language

df['Language'] = df['Review'].apply(do_language_identifying)
df['Language'].value_counts()

Language
en    20474
fr        7
nl        3
af        3
es        1
da        1
ro        1
it        1
Name: count, dtype: int64

Removing non-english reviews

In [7]:
df = df[df['Language'] == 'en']

### Tokenization / Removing Punctuation / Case Normalization

In [8]:
WORD_RE = regex.compile(r"(?V1)\p{L}+(?:[’'-]\p{L}+)*")

def tokenize_for_topics(text):
    text = unicodedata.normalize("NFKC", str(text)).lower()
    text = regex.sub(r"[‘’`´]", "'", text)      # normalize apostrophes
    text = regex.sub(r"[‐‑‒–—−]", "-", text)    # normalize dash variants
    return WORD_RE.findall(text)

df["Tokens"] = df["Review"].apply(tokenize_for_topics)

df['Tokens'][0]

['nice',
 'hotel',
 'expensive',
 'parking',
 'got',
 'good',
 'deal',
 'stay',
 'hotel',
 'anniversary',
 'arrived',
 'late',
 'evening',
 'took',
 'advice',
 'previous',
 'reviews',
 'did',
 'valet',
 'parking',
 'check',
 'quick',
 'easy',
 'little',
 'disappointed',
 'non-existent',
 'view',
 'room',
 'room',
 'clean',
 'nice',
 'size',
 'bed',
 'comfortable',
 'woke',
 'stiff',
 'neck',
 'high',
 'pillows',
 'not',
 'soundproof',
 'like',
 'heard',
 'music',
 'room',
 'night',
 'morning',
 'loud',
 'bangs',
 'doors',
 'opening',
 'closing',
 'hear',
 'people',
 'talking',
 'hallway',
 'maybe',
 'just',
 'noisy',
 'neighbors',
 'aveda',
 'bath',
 'products',
 'nice',
 'did',
 'not',
 'goldfish',
 'stay',
 'nice',
 'touch',
 'taken',
 'advantage',
 'staying',
 'longer',
 'location',
 'great',
 'walking',
 'distance',
 'shopping',
 'overall',
 'nice',
 'experience',
 'having',
 'pay',
 'parking',
 'night']

### Removing Stop Words

In [9]:
list_stop_words = nltk.corpus.stopwords.words("English")
df['Tokens'] = df['Tokens'].apply(lambda tokens: [token for token in tokens if token not in list_stop_words])

### Lemmatization

In [10]:
lemmatizer = WordNetLemmatizer()
df['Tokens'] = df['Tokens'].apply(lambda tokens: [lemmatizer.lemmatize(token) for token in tokens])

In [11]:
df['Tokens'].head()

0    [nice, hotel, expensive, parking, got, good, d...
1    [ok, nothing, special, charge, diamond, member...
2    [nice, room, experience, hotel, monaco, seattl...
3    [unique, great, stay, wonderful, time, hotel, ...
4    [great, stay, great, stay, went, seahawk, game...
Name: Tokens, dtype: object

Models: Develop code to first vectorize your text data, and then train at least six LDA and six NMF topic models on these vectors. Use clear section headings for each type of model. Record each set of hyperparameters (for both vectorization and the topic models) that you try, and find the perplexity, word-topic table, and document-topic table for each model. Present this information neatly and use it to select your best LDA and NMF models.

### Vectorizing the text data: LDA
### Hyperparameter Stack:
.....

In [12]:
df['clean_text'] = df['Tokens'].apply(lambda tokens: ' '.join(tokens))
df[['Tokens', 'clean_text']].head()

,Tokens,clean_text
0,"[nice, hotel, expensive, parking, got, good, d...",nice hotel expensive parking got good deal sta...
1,"[ok, nothing, special, charge, diamond, member...",ok nothing special charge diamond member hilto...
2,"[nice, room, experience, hotel, monaco, seattl...",nice room experience hotel monaco seattle good...
3,"[unique, great, stay, wonderful, time, hotel, ...",unique great stay wonderful time hotel monaco ...
4,"[great, stay, great, stay, went, seahawk, game...",great stay great stay went seahawk game awesom...


In [ ]:
number_features = 2000

cv = CountVectorizer(
    max_df=0.95,        
    min_df=2,           
    max_features=number_features
)
 
dtm_cv = cv.fit_transform(df['clean_text'])


In [14]:
print(dtm_cv[0])

<Compressed Sparse Row sparse matrix of dtype 'int64'
	with 61 stored elements and shape (1, 2000)>
  Coords	Values
  (0, 1133)	5
  (0, 842)	2
  (0, 618)	1
  (0, 1231)	3
  (0, 752)	1
  (0, 750)	1
  (0, 455)	1
  (0, 1672)	2
  (0, 66)	1
  (0, 89)	1
  (0, 943)	1
  (0, 593)	1
  (0, 1795)	1
  (0, 29)	1
  (0, 1327)	1
  (0, 1459)	1
  (0, 1883)	1
  (0, 302)	1
  (0, 1373)	1
  (0, 552)	1
  (0, 977)	1
  (0, 501)	1
  (0, 1144)	1
  (0, 1892)	1
  (0, 1479)	3
  :	:
  (0, 1098)	1
  (0, 998)	1
  (0, 517)	1
  (0, 1185)	1
  (0, 794)	1
  (0, 1249)	1
  (0, 1741)	1
  (0, 774)	1
  (0, 1048)	1
  (0, 1143)	1
  (0, 150)	1
  (0, 1343)	1
  (0, 1801)	1
  (0, 1737)	1
  (0, 27)	1
  (0, 1674)	1
  (0, 992)	1
  (0, 987)	1
  (0, 758)	1
  (0, 1912)	1
  (0, 510)	1
  (0, 1566)	1
  (0, 1206)	1
  (0, 619)	1
  (0, 1244)	1


### Vectorization for NMF
### Hyperparameter Stack:
.....

In [15]:
tfidf = TfidfVectorizer(
    max_df=0.95,
    min_df=2,
    max_features=number_features
)
 
dtm_tfidf = tfidf.fit_transform(df['clean_text'])  # sparse matrix (n_docs x n_words)

In [16]:
print(dtm_tfidf[0])

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 61 stored elements and shape (1, 2000)>
  Coords	Values
  (0, 1133)	0.3053605163099298
  (0, 842)	0.07321841244590628
  (0, 618)	0.10583622874772876
  (0, 1231)	0.3551040339985228
  (0, 752)	0.07891408832490991
  (0, 750)	0.0541333656589954
  (0, 455)	0.11217125624432793
  (0, 1672)	0.10301271786098348
  (0, 66)	0.15410959873180716
  (0, 89)	0.09567720253463985
  (0, 943)	0.1121061250855655
  (0, 593)	0.10538065188667077
  (0, 1795)	0.0970748214761273
  (0, 29)	0.14346614996004356
  (0, 1327)	0.13453386429462422
  (0, 1459)	0.08259342736264798
  (0, 1883)	0.149192012598151
  (0, 302)	0.08800266219515164
  (0, 1373)	0.12873791407978522
  (0, 552)	0.10252713808941542
  (0, 977)	0.07721828275843941
  (0, 501)	0.11739811519341918
  (0, 1144)	0.1310305954002947
  (0, 1892)	0.07827151602757652
  (0, 1479)	0.1063293898058834
  :	:
  (0, 1098)	0.09138945122338905
  (0, 998)	0.14158320253786155
  (0, 517)	0.09323534119777747
  (0, 11